# GOMS 주제별 통계 크롤링 — 전공계열별 노동시장 기준선 (P2_G1_kedi)

**목표**: 한국고용정보원 고용조사분석시스템(GOMS)의 [주제별 통계](https://analysis.keis.or.kr/gomsSubject.do?goIndex=3-1) 화면에서
경제활동상태(3) + 현재 일자리(18) + 근로소득(9) + 근로시간(9) = **39개 주제**의 공식 CSV(실제로는 `.xls` 확장자의 HTML 표)를
자동화 다운로드하고, 정규화된 long-format 데이터로 변환한다.

**중요 caveat (반드시 숙지)**
- GOMS는 **전년도 대졸자 약 1만8천 명의 표본조사**이며 2019년 졸업자 조사를 끝으로 **잠정 중단**된 자료다.
- 따라서 이 데이터는 **2024년 대학·학과 실적이 아니라, 과거(2007~2019) 전국 전공계열별 노동시장 구조 기준선(baseline proxy)** 으로만 사용해야 한다.
- 동일 전공계열의 모든 대학에 같은 GOMS 값이 반복 부여되므로, 대학별 실적의 독립적인 설명변수처럼 통계 해석(p-value 등)을 해서는 안 된다.
- 인용 시 "한국고용정보원 GOMS 분석시스템의 가중 표본분석 결과"로 출처를 명시한다 (사이트 자체가 공식 통계자료가 아님을 명시).

**산출물 위치**: `workbook/p2/p2_2/data/goms_subject_crawl/` (원본 CSV는 절대 수정하지 않고, 정제 결과는 `normalized/`에 별도 저장)


## Gate 0 — 사이트 구조 탐색 결과 요약

실제 브라우저 자동화(Playwright, headless Chromium)로 페이지를 열어 확인한 결과:

- 겉보기엔 R Shiny 스타일 CSS(`shiny-bound-input` 클래스, `shiny.min.js`)를 쓰지만, 실제 데이터 호출은 **`POST /gomsSubjectAct.do`** 단일 엔드포인트를 쓰는
  일반 jQuery 기반 트리메뉴 앱이다. (Shiny 웹소켓은 404로 죽어있음 — 순수 장식용 CSS 잔재로 추정)
- 39개 주제는 좌측 트리메뉴에서 그룹 `data-tt-id`: `2`(경제활동 상태, 3개) · `3`(현재 일자리, 18개) · `4`(근로소득, 9개) · `5`(근로시간, 9개)에 분포한다.
- "CSV 다운로드" 버튼(`#cvsDownBtn`)은 실제로는 **HTML 표를 `.xls` 확장자로 감싼 파일**을 내려준다 (진짜 CSV/XLS 바이너리가 아님). 파싱은 `pandas.read_html`로 처리한다.
  단, 원본 마크업에 `colspan=&quot;1&quot;` 처럼 속성값이 이중 이스케이프되어 있어 `&quot;` → `"` 치환 전처리가 필요하다.

### 자동화 중 실제로 발견/수정한 버그 (재현 가능, 근거: 소스코드 + 해시 비교)

1. **다운로드가 재사용되는 팝업 창에서 발생함**: `#cvsDownBtn` 클릭은 새 팝업(`/excel.do`)을 열거나 기존 팝업을 재사용하는데,
   처음에는 메인 페이지에서 `download` 이벤트가 잡히지만 두 번째부터는 그 팝업에서만 발생한다.
   → 메인 페이지 + 알려진 모든 팝업에 대해 동시에 `download` 이벤트를 기다리는 방식으로 해결.
2. **표준편차/분산이 항상 평균으로 되돌아가는 사이트 자체 버그**: 페이지 내장 스크립트의 `callbackGOMSSubject()` 함수가
   연속형(그룹 4·5) 주제에서 `$("#viewType2 option").eq(0).prop("selected", true);` 를 **조건 없이** 실행한다
   (카테고리형은 `if (old_tg != tg)` 가드가 있지만 연속형엔 없음). 즉 "표준편차"를 선택해도 검색 버튼을 누르는 순간 항상 "평균"으로
   리셋되어, **정상적인 UI 흐름으로는 표준편차/분산을 절대 받을 수 없다.** → 실제 다운로드 파일 57개 중 34개가 서로 해시 중복이었던
   근본 원인. 이번 산출물에서는 **표준편차를 제외**하고 이 사실을 QA 요약에 기록했다.
3. **초기 페이지 로드시 기본 주제(`goIndex=3-1`, 취업자의 성별 산업 분포) 응답이 늦게 도착해 이후 선택한 주제를 덮어쓰는 경쟁 상태**:
   빠르게 반복 실행할 때 헤더 구조(연속형은 1행, 범주형은 3행)로 검증하지 않으면 다른 주제의 데이터가 섞여 들어간다.
   → 클릭 직후 항상 `#tresult thead tr` 개수를 확인해 연속형이 1행으로 안정될 때까지 대기·재시도하도록 수정.
4. **연속형 주제는 `#weighted` 체크박스 자체가 숨겨져 있음** (`$(".weighted").hide()`), `force=True`로도 클릭 불가 —
   연속형 통계는 애초에 가중치 옵션이 없고 항상 모집단 가중 평균으로 계산되는 것으로 확인되어 해당 단계를 제거했다.


In [1]:
import asyncio
import hashlib
import re
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from playwright.async_api import async_playwright

BASE_URL = "https://analysis.keis.or.kr/gomsSubject.do?goIndex=3-1"
ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/goms_subject_crawl")
RAW_DIR = ROOT / "raw_downloads"
SNAPSHOT_DIR = ROOT / "page_snapshots"
LOG_DIR = ROOT / "logs"
NORM_DIR = ROOT / "normalized"
QA_DIR = ROOT / "qa"

for d in [RAW_DIR / "frequency_share", RAW_DIR / "mean", SNAPSHOT_DIR / "html",
          SNAPSHOT_DIR / "screenshots", LOG_DIR, NORM_DIR, QA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def safe_slug(text):
    text = re.sub(r"[()~]", "", text)
    text = re.sub(r"\s+", "_", text.strip())
    return text


In [2]:
# 39개 주제 레지스트리 — 그룹 2(경제활동 상태) + 3(현재 일자리) + 4(근로소득) + 5(근로시간)
TOPICS = [
    {"group": "2", "group_name": "경제활동 상태", "topic": "성별 경제활동 상태", "kind": "categorical"},
    {"group": "2", "group_name": "경제활동 상태", "topic": "학교유형별 경제활동 상태", "kind": "categorical"},
    {"group": "2", "group_name": "경제활동 상태", "topic": "전공계열별 경제활동 상태", "kind": "categorical"},

    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 성별 산업 분포", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 학교유형별 산업 분포", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 전공별 산업 분포", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 성별 직업 분포(~2016)", "kind": "categorical", "classification_version": "pre_2017"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 성별 직업 분포(2017~)", "kind": "categorical", "classification_version": "post_2017"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 학교유형별 직업 분포(~2016)", "kind": "categorical", "classification_version": "pre_2017"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 학교유형별 직업 분포(2017~)", "kind": "categorical", "classification_version": "post_2017"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 전공별 직업 분포(~2016)", "kind": "categorical", "classification_version": "pre_2017"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 전공별 직업 분포(2017~)", "kind": "categorical", "classification_version": "post_2017"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 성별 사업체 규모", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 학교유형별 사업체 규모", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 전공별 사업체 규모", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 성별 사업체 유형", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 학교유형별 사업체 유형", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 전공별 사업체 유형", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 성별 종사상 지위", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 학교유형별 종사상 지위", "kind": "categorical"},
    {"group": "3", "group_name": "현재 일자리", "topic": "취업자의 전공별 종사상 지위", "kind": "categorical"},

    {"group": "4", "group_name": "근로소득", "topic": "성별 월평균 근로소득", "kind": "continuous"},
    {"group": "4", "group_name": "근로소득", "topic": "학교유형별 월평균 근로소득", "kind": "continuous"},
    {"group": "4", "group_name": "근로소득", "topic": "전공계열별 월평균 근로소득", "kind": "continuous"},
    {"group": "4", "group_name": "근로소득", "topic": "산업별 월평균 근로소득", "kind": "continuous"},
    {"group": "4", "group_name": "근로소득", "topic": "직업별 월평균 근로소득(~2016)", "kind": "continuous", "classification_version": "pre_2017"},
    {"group": "4", "group_name": "근로소득", "topic": "직업별 월평균 근로소득(2017~)", "kind": "continuous", "classification_version": "post_2017"},
    {"group": "4", "group_name": "근로소득", "topic": "사업체 규모별 월평균 근로소득", "kind": "continuous"},
    {"group": "4", "group_name": "근로소득", "topic": "사업체 유형별 월평균 근로소득", "kind": "continuous"},
    {"group": "4", "group_name": "근로소득", "topic": "종사상지위별 월평균 근로소득", "kind": "continuous"},

    {"group": "5", "group_name": "근로시간", "topic": "성별 주당 평균 근로시간", "kind": "continuous"},
    {"group": "5", "group_name": "근로시간", "topic": "학교유형별 주당 평균 근로시간", "kind": "continuous"},
    {"group": "5", "group_name": "근로시간", "topic": "전공계열별 주당 평균 근로시간", "kind": "continuous"},
    {"group": "5", "group_name": "근로시간", "topic": "산업별 주당 평균 근로시간", "kind": "continuous"},
    {"group": "5", "group_name": "근로시간", "topic": "직업별 주당 평균 근로시간(~2016)", "kind": "continuous", "classification_version": "pre_2017"},
    {"group": "5", "group_name": "근로시간", "topic": "직업별 주당 평균 근로시간(2017~)", "kind": "continuous", "classification_version": "post_2017"},
    {"group": "5", "group_name": "근로시간", "topic": "사업체 규모별 주당 평균 근로시간", "kind": "continuous"},
    {"group": "5", "group_name": "근로시간", "topic": "사업체 유형별 주당 평균 근로시간", "kind": "continuous"},
    {"group": "5", "group_name": "근로시간", "topic": "종사상지위별 주당 평균 근로시간", "kind": "continuous"},
]
for i, t in enumerate(TOPICS, start=1):
    t["topic_id"] = f"GOMS_{i:03d}"
    t.setdefault("classification_version", "all")

assert len(TOPICS) == 39
topic_registry_df = pd.DataFrame(TOPICS)
topic_registry_df.to_csv(ROOT / "00_topic_registry.csv", index=False, encoding="utf-8-sig")
topic_registry_df.groupby("group_name").size()


group_name
경제활동 상태     3
근로소득        9
근로시간        9
현재 일자리     18
dtype: int64

## 크롤러 핵심 함수

아래 함수들은 실제로 이번 세션에서 사이트를 상대로 검증·수정을 거친 최종 버전이다.


In [3]:
async def ensure_group_expanded(page, group_id):
    row = page.locator(f'tr[data-tt-id="{group_id}"]')
    cls = await row.get_attribute("class")
    if cls and "collapsed" in cls:
        await row.locator(".indenter a").click()
        await page.wait_for_timeout(400)

async def click_topic(page, group_id, topic_text):
    await ensure_group_expanded(page, group_id)
    link = page.locator(f'tr[data-tt-parent-id="{group_id}"] a', has_text=topic_text)
    await link.first.wait_for(state="visible", timeout=8000)
    await link.first.click()
    await page.wait_for_timeout(1200)

async def set_full_year_range(page):
    syear_opts = await page.locator("#syear option").evaluate_all("els => els.map(e => e.value)")
    eyear_opts = await page.locator("#eyear option").evaluate_all("els => els.map(e => e.value)")
    start, end = min(syear_opts), max(eyear_opts)
    await page.locator("#syear").select_option(start)
    await page.locator("#eyear").select_option(end)
    return start, end

async def download_csv(page, known_popups, out_path, timeout_ms=15000):
    """메인 페이지 + 이미 알려진 모든 팝업에서 동시에 download 이벤트를 기다린다 (팝업 재사용 버그 대응)."""
    tasks = [asyncio.ensure_future(page.wait_for_event("download", timeout=timeout_ms))]
    for p in known_popups:
        if not p.is_closed():
            tasks.append(asyncio.ensure_future(p.wait_for_event("download", timeout=timeout_ms)))
    await page.locator("#cvsDownBtn").click()
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED, timeout=timeout_ms/1000 + 1)
    download = None
    for d in done:
        try:
            download = d.result()
            break
        except Exception:
            continue
    for p in pending:
        p.cancel()
    if download is None:
        raise TimeoutError("no download event from page or known popups")
    await download.save_as(str(out_path))
    return download.suggested_filename

async def wait_for_single_row_header(page, timeout_s=8):
    """연속형 주제는 thead가 1행이어야 정상 (다중 행이면 초기 기본 주제가 덮어쓴 상태)."""
    for _ in range(timeout_s * 2):
        n = await page.evaluate("() => document.querySelectorAll('#tresult thead tr').length")
        if n == 1:
            return True
        await page.wait_for_timeout(500)
    return False


## 라이브 데모 — 대표 2개 주제로 크롤러 동작 검증

전체 39개 주제(57회 다운로드 시도, 버그 수정 재시도 포함) 크롤링은 실제로 15\~25분이 걸리고 외부 정부 서버에 반복 요청을 보내므로,
이 노트북을 열 때마다 재실행하지 않는다. 아래 셀은 위 함수들이 **지금도 실제로 동작함**을 대표 주제 2개(범주형 1 + 연속형 1)로
라이브 검증한다.


In [4]:
async def demo_fetch(group_id, topic_text, kind):
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context(locale="ko-KR", accept_downloads=True)
        known_popups = []
        context.on("page", lambda p: known_popups.append(p))
        page = await context.new_page()
        await page.goto(BASE_URL, wait_until="networkidle", timeout=45000)
        await page.wait_for_timeout(1500)

        await click_topic(page, group_id, topic_text)
        if kind == "categorical":
            weighted = page.locator("#weighted")
            if not await weighted.is_checked():
                await weighted.check(force=True)
            await page.locator("#viewType1").select_option(label="빈도+비중")
        start, end = await set_full_year_range(page)
        await page.locator("#searchBtn").click()
        await page.wait_for_timeout(1800)
        if kind == "continuous":
            await wait_for_single_row_header(page, timeout_s=8)

        out_path = Path("/tmp") / f"demo_{safe_slug(topic_text)}.xls"
        suggested = await download_csv(page, known_popups, out_path, timeout_ms=15000)
        data = out_path.read_bytes()
        await browser.close()
        return {
            "topic": topic_text, "kind": kind, "year_range": (start, end),
            "file_size": len(data), "sha256": sha256_bytes(data)[:16],
            "suggested_filename": suggested,
        }

demo_results = []
demo_results.append(await demo_fetch("2", "전공계열별 경제활동 상태", "categorical"))
demo_results.append(await demo_fetch("4", "전공계열별 월평균 근로소득", "continuous"))
pd.DataFrame(demo_results)


,topic,kind,year_range,file_size,sha256,suggested_filename
0,전공계열별 경제활동 상태,categorical,"(2007, 2019)",42140,2dc61cf9b42cfa0e,data.xls
1,전공계열별 월평균 근로소득,continuous,"(2007, 2019)",6354,f99385fb389768be,data.xls


라이브 데모 다운로드 두 건이 서로 다른 `sha256`과 정상적인 `file_size`를 갖는다면, 팝업 다운로드 추적과 연속형 헤더 검증 로직이
현재도 정상 동작함을 의미한다.


## 전체 39개 주제(최종 유효 CSV) 크롤링 결과 불러오기

전체 크롤링은 이번 세션에서 백그라운드로 3차에 걸쳐 실행되었다 (자세한 실패/재시도 이력은 `logs/02b~02d_*.csv` 참고):

1. **1차 전체 실행 (57개 시도: 범주형 21 + 연속형 mean 18 + stddev 18)**: 팝업 다운로드 버그를 고친 뒤 실행 → 범주형 21개는 전부 정상,
   연속형 18개는 위 버그 3·4번(경쟁 상태, 숨겨진 weighted 체크박스)으로 인해 다수 오염 (해시 비교로 34/36건 중복 확인).
2. **2~3차 연속형 재수집**: 헤더 구조 검증 + weighted 단계 제거 + 응답 대기 타임아웃 확대로 18개 전부 재수집, 최종 해시 중복 0건 확인.
3. **표준편차는 사이트 자체 버그로 정상 취득이 불가능함을 소스코드 레벨에서 확인**하여 최종 산출물에서 제외했다 (아래 QA 요약 참고).

최종적으로 `02_download_manifest.csv`는 **39/39 성공, 파일 해시 중복 0건**의 검증된 상태다.


In [5]:
manifest = pd.read_csv(ROOT / "02_download_manifest.csv")
print(f"총 {len(manifest)}개 주제, kind별:")
print(manifest["kind"].value_counts())
print(f"\n상태별:")
print(manifest["status"].value_counts())
print(f"\n중복 sha256 여부:", manifest.groupby("sha256").filter(lambda g: len(g) > 1).empty == False and "있음" or "없음")
manifest.head(10)


총 39개 주제, kind별:
kind
categorical    21
continuous     18
Name: count, dtype: int64

상태별:
status
success    39
Name: count, dtype: int64

중복 sha256 여부: 없음


,topic_id,topic_group,topic_name,measure,classification_version,kind,status,file_path,file_size,sha256
0,GOMS_001,경제활동 상태,성별 경제활동 상태,freq_share,all,categorical,success,raw_downloads/frequency_share/GOMS_001_성별_경제활동...,19274,759f740ac89cb419c9d60f9aeb12864ba98f59cc9f5ab2...
1,GOMS_002,경제활동 상태,학교유형별 경제활동 상태,freq_share,all,categorical,success,raw_downloads/frequency_share/GOMS_002_학교유형별_경...,23747,f90310a392127ba4d0332de3c151b4f5f209c57ae66489...
2,GOMS_003,경제활동 상태,전공계열별 경제활동 상태,freq_share,all,categorical,success,raw_downloads/frequency_share/GOMS_003_전공계열별_경...,42140,2dc61cf9b42cfa0e1bec2d85f26ffd66561add6167367e...
3,GOMS_004,현재 일자리,취업자의 성별 산업 분포,freq_share,all,categorical,success,raw_downloads/frequency_share/GOMS_004_취업자의_성별...,79875,ecfe173c6d19e2b0951cf295bc38bc002dd87484eafb44...
4,GOMS_005,현재 일자리,취업자의 학교유형별 산업 분포,freq_share,all,categorical,success,raw_downloads/frequency_share/GOMS_005_취업자의_학교...,104437,b46fc4433a5f101ce1e8c15ab9e12b41ec8526457eff47...
5,GOMS_006,현재 일자리,취업자의 전공별 산업 분포,freq_share,all,categorical,success,raw_downloads/frequency_share/GOMS_006_취업자의_전공...,205090,be2a385485d2016b2ba798e0a884863af33bf9bfe51fc2...
6,GOMS_007,현재 일자리,취업자의 성별 직업 분포(~2016),freq_share,pre_2017,categorical,success,raw_downloads/frequency_share/GOMS_007_취업자의_성별...,70066,e116579e757be865b0387cb10f7e14e8a6e6236013d6ec...
7,GOMS_008,현재 일자리,취업자의 성별 직업 분포(2017~),freq_share,post_2017,categorical,success,raw_downloads/frequency_share/GOMS_008_취업자의_성별...,11379,e79e4f957851be797035a012e282fa55e1fdd320d007ab...
8,GOMS_009,현재 일자리,취업자의 학교유형별 직업 분포(~2016),freq_share,pre_2017,categorical,success,raw_downloads/frequency_share/GOMS_009_취업자의_학교...,91509,8d3d879391c56e3949a309b7840ebbcdb041270cbd3f23...
9,GOMS_010,현재 일자리,취업자의 학교유형별 직업 분포(2017~),freq_share,post_2017,categorical,success,raw_downloads/frequency_share/GOMS_010_취업자의_학교...,14346,cfb8637aec9128431ee0c1beb0adfda4179804bb4b164a...


## 정규화 (Wide HTML 표 → Long format)

In [6]:
def load_table(path):
    raw = path.read_text(encoding="utf-8-sig")
    fixed = raw.replace("&quot;", '"')
    return pd.read_html(StringIO(fixed))[0]

def parse_categorical(path, topic_meta):
    df = load_table(path)
    new_cols = list(df.columns)
    new_cols[0] = ("dim", "dim", "dim")
    df.columns = pd.MultiIndex.from_tuples(new_cols)
    df = df.set_index(("dim", "dim", "dim"))
    df.index.name = "dimension_value"

    records = []
    for (year, subgroup, measure_kr), col in df.items():
        if year == "Unnamed: 0_level_0":
            continue
        measure_type = "frequency" if measure_kr == "빈도" else "share"
        for dim_val, val in col.items():
            records.append({
                "topic_id": topic_meta["topic_id"], "topic_group": topic_meta["group_name"],
                "topic_name": topic_meta["topic"], "classification_version": topic_meta["classification_version"],
                "dimension_value": dim_val, "year": int(year), "subgroup": subgroup,
                "measure_type": measure_type,
                "value": pd.to_numeric(str(val).replace(",", ""), errors="coerce"),
            })
    return pd.DataFrame(records)

def parse_continuous(path, topic_meta):
    df = load_table(path)
    df = df.rename(columns={df.columns[0]: "dimension_value"})
    long = df.melt(id_vars="dimension_value", var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    long["topic_id"] = topic_meta["topic_id"]
    long["topic_group"] = topic_meta["group_name"]
    long["topic_name"] = topic_meta["topic"]
    long["classification_version"] = topic_meta["classification_version"]
    long["measure_type"] = "mean_income_or_hours"
    return long[["topic_id", "topic_group", "topic_name", "classification_version",
                 "dimension_value", "year", "measure_type", "value"]]


In [7]:
reg = pd.read_csv(ROOT / "00_topic_registry.csv").set_index("topic_id")

audit_rows, cat_frames, cont_frames, parse_issues = [], [], [], []

for _, row in manifest.iterrows():
    path = ROOT / row["file_path"]
    meta = reg.loc[row["topic_id"]].to_dict()
    meta["topic_id"] = row["topic_id"]
    try:
        if row["kind"] == "categorical":
            long_df = parse_categorical(path, meta)
            cat_frames.append(long_df)
        else:
            long_df = parse_continuous(path, meta)
            cont_frames.append(long_df)
        audit_rows.append({
            "topic_id": row["topic_id"], "topic_name": row["topic_name"], "kind": row["kind"],
            "n_records": len(long_df), "n_years": long_df["year"].nunique(),
            "year_min": long_df["year"].min(), "year_max": long_df["year"].max(),
            "n_dimension_values": long_df["dimension_value"].nunique(),
            "n_null_values": long_df["value"].isna().sum(), "parse_status": "ok",
        })
    except Exception as exc:
        parse_issues.append({"topic_id": row["topic_id"], "topic_name": row["topic_name"],
                              "error_type": type(exc).__name__, "error_message": str(exc)[:300]})
        audit_rows.append({"topic_id": row["topic_id"], "topic_name": row["topic_name"],
                            "kind": row["kind"], "parse_status": "failed"})

pd.DataFrame(audit_rows).to_csv(ROOT / "03_schema_audit.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(parse_issues).to_csv(ROOT / "04_parse_issues.csv", index=False, encoding="utf-8-sig")

cat_long = pd.concat(cat_frames, ignore_index=True)
cont_long = pd.concat(cont_frames, ignore_index=True)
cat_long.to_csv(NORM_DIR / "goms_distribution_long.csv", index=False, encoding="utf-8-sig")
cat_long.to_parquet(NORM_DIR / "goms_distribution_long.parquet", index=False)
cont_long.to_csv(NORM_DIR / "goms_continuous_long.csv", index=False, encoding="utf-8-sig")
cont_long.to_parquet(NORM_DIR / "goms_continuous_long.parquet", index=False)

print("파싱 실패:", len(parse_issues))
print("categorical long rows:", len(cat_long), " / continuous long rows:", len(cont_long))
cat_long.head()


파싱 실패: 0
categorical long rows: 29160  / continuous long rows: 2230


,topic_id,topic_group,topic_name,classification_version,dimension_value,year,subgroup,measure_type,value
0,GOMS_001,경제활동 상태,성별 경제활동 상태,all,전체,2007,전체,frequency,498699.0
1,GOMS_001,경제활동 상태,성별 경제활동 상태,all,남자,2007,전체,frequency,249657.0
2,GOMS_001,경제활동 상태,성별 경제활동 상태,all,여자,2007,전체,frequency,249041.0
3,GOMS_001,경제활동 상태,성별 경제활동 상태,all,전체,2007,전체,share,100.0
4,GOMS_001,경제활동 상태,성별 경제활동 상태,all,남자,2007,전체,share,50.0


In [8]:
cont_long.head()


,topic_id,topic_group,topic_name,classification_version,dimension_value,year,measure_type,value
0,GOMS_022,근로소득,성별 월평균 근로소득,all,전체,2007,mean_income_or_hours,195.8
1,GOMS_022,근로소득,성별 월평균 근로소득,all,남자,2007,mean_income_or_hours,216.5
2,GOMS_022,근로소득,성별 월평균 근로소득,all,여자,2007,mean_income_or_hours,167.0
3,GOMS_022,근로소득,성별 월평균 근로소득,all,전체,2008,mean_income_or_hours,188.9
4,GOMS_022,근로소득,성별 월평균 근로소득,all,남자,2008,mean_income_or_hours,208.1


## QA 검증

In [9]:
share = cat_long[cat_long["measure_type"] == "share"]
share_no_total = share[share["dimension_value"] != "전체"]
sums = (share_no_total.groupby(["topic_id", "topic_name", "year", "subgroup"])["value"]
        .sum().reset_index(name="share_sum"))
sums["qa_status"] = np.where(sums["share_sum"].between(90.0, 101.5), "PASS", "REVIEW")
sums.to_csv(QA_DIR / "proportion_sum_check.csv", index=False, encoding="utf-8-sig")

year_cov = manifest.merge(
    pd.concat([
        cat_long.groupby("topic_id")["year"].agg(["min", "max", "nunique"]),
        cont_long.groupby("topic_id")["year"].agg(["min", "max", "nunique"]),
    ]).reset_index(),
    on="topic_id", how="left"
)[["topic_id", "topic_name", "min", "max", "nunique"]]
year_cov.columns = ["topic_id", "topic_name", "year_min", "year_max", "n_years"]
year_cov["qa_status"] = np.where(year_cov["n_years"] >= 1, "PASS", "REVIEW")
year_cov.to_csv(QA_DIR / "year_coverage_check.csv", index=False, encoding="utf-8-sig")

cat_dup = cat_long.duplicated(subset=["topic_id", "dimension_value", "year", "subgroup", "measure_type"]).sum()
cont_dup = cont_long.duplicated(subset=["topic_id", "dimension_value", "year"]).sum()
pd.DataFrame([
    {"dataset": "categorical", "duplicate_rows": int(cat_dup)},
    {"dataset": "continuous", "duplicate_rows": int(cont_dup)},
]).to_csv(QA_DIR / "duplicate_check.csv", index=False, encoding="utf-8-sig")

cross_dup = manifest.groupby("sha256").filter(lambda g: len(g) > 1)

final_summary = pd.DataFrame([{
    "total_topics": len(manifest),
    "categorical_topics": (manifest["kind"] == "categorical").sum(),
    "continuous_topics": (manifest["kind"] == "continuous").sum(),
    "parse_failures": len(parse_issues),
    "proportion_review_count": (sums["qa_status"] == "REVIEW").sum(),
    "year_coverage_review_count": (year_cov["qa_status"] == "REVIEW").sum(),
    "duplicate_rows_categorical": int(cat_dup),
    "duplicate_rows_continuous": int(cont_dup),
    "cross_file_hash_duplicates": "PASS" if cross_dup.empty else "REVIEW",
    "stddev_available": False,
    "stddev_reason": "site JS bug (callbackGOMSSubject unconditionally resets viewType2 to index 0 on every render) makes 분산/표준편차 unreachable via official UI/CSV flow",
}])
final_summary.to_csv(QA_DIR / "final_qa_summary.csv", index=False, encoding="utf-8-sig")
final_summary.T


,0
total_topics,39
categorical_topics,21
continuous_topics,18
parse_failures,0
proportion_review_count,0
year_coverage_review_count,0
duplicate_rows_categorical,0
duplicate_rows_continuous,0
cross_file_hash_duplicates,PASS
stddev_available,False


## 요약

- **39/39 주제 CSV 수집 성공**, 정규화 후 파싱 실패 0건, 비중 합계(90\~101.5% 허용) 전량 PASS, 중복 행 0건, 파일 해시 중복 0건.
- **표준편차/분산은 사이트 자체의 JS 버그로 취득 불가**함을 소스코드 레벨에서 확인하여 이번 산출물에서 제외했다 (평균만 제공).
- 이 데이터는 **2007~2019년 GOMS 표본조사 기준**이며, **2024년 대학·학과 실적이 아니라 과거 전국 전공계열별 노동시장 구조의 기준선**으로만
  2024년 프로젝트에 통제변수/맥락자료로 결합해야 한다 (동일 전공계열의 모든 대학에 동일 값이 반복 부여되므로 독립 설명변수로 오용 금지).
- 산출물: `00_topic_registry.csv`, `02_download_manifest.csv`, `03_schema_audit.csv`, `04_parse_issues.csv`,
  `raw_downloads/{frequency_share,mean}/`, `normalized/goms_distribution_long.{csv,parquet}`,
  `normalized/goms_continuous_long.{csv,parquet}`, `qa/*.csv`, `logs/*` (요청/응답 로그 및 디버깅 이력).


## Gate 4 — raw_downloads 39개 파일 순차 기술통계·시각화

대상 폴더는 `workbook/p2/p2_2/final/final_data/raw_downloads/`이며, 아래는 정렬된 `GOMS_001`~`GOMS_039` 원본 HTML-XLS 파일을 파일 단위로 확인하는 섹션이다.

각 파일마다 다음을 붙였다.

- 원본 표 크기, long 변환 행수, 연도 범위, 하위그룹/항목 수
- 수치형 기술통계와 최근 연도 상위값
- 파일별 PNG 시각화
- 바로 재실행할 수 있는 코드 셀 `show_goms_raw_file(...)`

생성된 그림/표는 `workbook/p2/p2_2/final/eda/P2_G1_kedi_raw_download_visuals`에 저장했다.


In [ ]:
from pathlib import Path
from io import StringIO
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

RAW_DOWNLOAD_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/final/final_data/raw_downloads")

def _read_goms_html_xls(path):
    raw = path.read_text(encoding="utf-8-sig")
    return pd.read_html(StringIO(raw.replace("&quot;", '"')))[0]

def _parse_goms_raw(path):
    path = Path(path)
    if path.parent.name == "frequency_share":
        raw = _read_goms_html_xls(path)
        cols = list(raw.columns)
        cols[0] = ("dimension_value", "dimension_value", "dimension_value")
        raw.columns = pd.MultiIndex.from_tuples(cols)
        raw = raw.set_index(("dimension_value", "dimension_value", "dimension_value"))
        raw.index.name = "dimension_value"
        records = []
        for (year, subgroup, measure_kr), series in raw.items():
            if str(year).startswith("Unnamed"):
                continue
            measure_type = "frequency" if measure_kr == "빈도" else "share"
            for dimension_value, value in series.items():
                records.append({
                    "year": int(year),
                    "subgroup": subgroup,
                    "dimension_value": dimension_value,
                    "measure_type": measure_type,
                    "value": pd.to_numeric(str(value).replace(",", ""), errors="coerce"),
                })
        return raw.reset_index(), pd.DataFrame(records)
    raw = _read_goms_html_xls(path)
    raw = raw.rename(columns={raw.columns[0]: "dimension_value"})
    long = raw.melt(id_vars="dimension_value", var_name="year", value_name="value")
    long["year"] = long["year"].astype(int)
    long["subgroup"] = "평균"
    long["measure_type"] = "mean"
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    return raw, long[["year", "subgroup", "dimension_value", "measure_type", "value"]]

def show_goms_raw_file(relative_file_path, top_n=12):
    path = RAW_DOWNLOAD_ROOT / relative_file_path
    raw, long = _parse_goms_raw(path)
    display(Markdown(f"**파일**: `{relative_file_path}`  \n**원본 표 크기**: {raw.shape[0]:,}행 x {raw.shape[1]:,}열  \n**정규화 미리보기 행수**: {len(long):,}행"))
    display(long.groupby("measure_type")["value"].describe().round(2))
    display(raw.head(8))

    latest_year = int(long["year"].max())
    if path.parent.name == "frequency_share":
        plot_df = long[
            (long["year"] == latest_year)
            & (long["measure_type"] == "share")
            & (long["dimension_value"] != "전체")
        ].copy()
        pivot = plot_df.pivot_table(index="dimension_value", columns="subgroup", values="value", aggfunc="sum").fillna(0)
        if len(pivot) > top_n:
            keep = pivot.max(axis=1).sort_values(ascending=False).head(top_n).index
            pivot = pivot.loc[keep]
        display(Markdown(f"**{latest_year}년 비중 상위/주요 항목 heatmap**"))
        fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 0.8 + 4), max(4, len(pivot) * 0.35 + 2)))
        im = ax.imshow(pivot.values, aspect="auto", cmap="YlGnBu")
        ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha="right")
        ax.set_yticks(range(len(pivot.index)), pivot.index)
        ax.set_xlabel("하위그룹")
        ax.set_ylabel("항목")
        fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="비중(%)")
        fig.tight_layout()
        plt.show()
    else:
        latest = long[(long["year"] == latest_year) & (long["dimension_value"] != "전체")].sort_values("value", ascending=False).head(top_n)
        fig, ax = plt.subplots(figsize=(9, max(4, len(latest) * 0.35 + 1.5)))
        bar = latest.sort_values("value", ascending=True)
        ax.barh(bar["dimension_value"], bar["value"], color="#497a8f")
        ax.set_title(f"{latest_year}년 평균값 상위 {len(bar)}개")
        ax.set_xlabel("평균값")
        fig.tight_layout()
        plt.show()


### GOMS_001 — 성별 경제활동 상태

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_001_성별_경제활동_상태_freq_share.xls` |
| 원본 표 크기 | 3행 x 105열 |
| long 변환 행수 | 312행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 3 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 156 | 12265.0 | 135216.5 | 165244.7 | 520547.0 |
| share | 156 | 41.7 | 51.6 | 66.6 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 비경활 | 여자 | 54.6 |
| 2019 | 전체 | 여자 | 51.3 |
| 2019 | 취업자 | 여자 | 50.5 |
| 2019 | 실업자 | 남자 | 50.4 |
| 2019 | 실업자 | 여자 | 49.5 |

![GOMS_001 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_001_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_001_성별_경제활동_상태_freq_share.xls")


### GOMS_002 — 학교유형별 경제활동 상태

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_002_학교유형별_경제활동_상태_freq_share.xls` |
| 원본 표 크기 | 4행 x 105열 |
| long 변환 행수 | 416행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 4 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 208 | 28.0 | 43660.5 | 123933.4 | 520546.0 |
| share | 208 | 0.0 | 49.8 | 50.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 실업자 | 4년제 | 69.1 |
| 2019 | 전체 | 4년제 | 65.3 |
| 2019 | 취업자 | 4년제 | 65.0 |
| 2019 | 비경활 | 4년제 | 64.4 |
| 2019 | 비경활 | 전문대 | 35.1 |

![GOMS_002 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_002_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_002_학교유형별_경제활동_상태_freq_share.xls")


### GOMS_003 — 전공계열별 경제활동 상태

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_003_전공계열별_경제활동_상태_freq_share.xls` |
| 원본 표 크기 | 8행 x 105열 |
| long 변환 행수 | 832행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 8 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 416 | 585.0 | 30691.0 | 61966.5 | 520547.0 |
| share | 416 | 1.7 | 12.6 | 25.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 실업자 | 사회 | 30.7 |
| 2019 | 실업자 | 공학 | 26.5 |
| 2019 | 전체 | 사회 | 26.1 |
| 2019 | 비경활 | 사회 | 25.8 |
| 2019 | 취업자 | 사회 | 25.5 |

![GOMS_003 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_003_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_003_전공계열별_경제활동_상태_freq_share.xls")


### GOMS_004 — 취업자의 성별 산업 분포

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_004_취업자의_성별_산업_분포_freq_share.xls` |
| 원본 표 크기 | 22행 x 79열 |
| long 변환 행수 | 1,716행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 3 |
| 항목 수 | 22 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 858 | 0.0 | 7553.0 | 22563.2 | 392483.0 |
| share | 858 | 0.0 | 3.3 | 9.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 여자 | 보건업 및 사회복지 서비스업 | 25.5 |
| 2019 | 남자 | 제조업 | 20.7 |
| 2019 | 전체 | 보건업 및 사회복지 서비스업 | 16.7 |
| 2019 | 여자 | 교육 서비스업 | 15.0 |
| 2019 | 전체 | 제조업 | 14.5 |

![GOMS_004 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_004_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_004_취업자의_성별_산업_분포_freq_share.xls")


### GOMS_005 — 취업자의 학교유형별 산업 분포

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_005_취업자의_학교유형별_산업_분포_freq_share.xls` |
| 원본 표 크기 | 22행 x 105열 |
| long 변환 행수 | 2,288행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 22 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 1,144 | 0.0 | 3747.0 | 16922.4 | 392483.0 |
| share | 1,144 | 0.0 | 2.0 | 9.1 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 교육대 | 교육 서비스업 | 98.8 |
| 2019 | 전문대 | 보건업 및 사회복지 서비스업 | 27.3 |
| 2019 | 전체 | 보건업 및 사회복지 서비스업 | 16.7 |
| 2019 | 4년제 | 제조업 | 14.8 |
| 2019 | 전문대 | 제조업 | 14.5 |

![GOMS_005 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_005_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_005_취업자의_학교유형별_산업_분포_freq_share.xls")


### GOMS_006 — 취업자의 전공별 산업 분포

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_006_취업자의_전공별_산업_분포_freq_share.xls` |
| 원본 표 크기 | 22행 x 209열 |
| long 변환 행수 | 4,576행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 8 |
| 항목 수 | 22 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 2,288 | 0.0 | 1164.5 | 8460.9 | 392483.0 |
| share | 2,288 | 0.0 | 2.1 | 9.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 의약 | 보건업 및 사회복지 서비스업 | 80.2 |
| 2019 | 교육 | 교육 서비스업 | 60.4 |
| 2019 | 공학 | 제조업 | 29.7 |
| 2019 | 교육 | 보건업 및 사회복지 서비스업 | 22.0 |
| 2019 | 자연 | 제조업 | 19.7 |

![GOMS_006 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_006_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_006_취업자의_전공별_산업_분포_freq_share.xls")


### GOMS_007 — 취업자의 성별 직업 분포2016

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_007_취업자의_성별_직업_분포2016_freq_share.xls` |
| 원본 표 크기 | 25행 x 61열 |
| long 변환 행수 | 1,500행 |
| 연도 범위 | 2007~2016 (10개 연도) |
| 하위그룹 수 | 3 |
| 항목 수 | 25 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 750 | 62.0 | 4727.0 | 20158.3 | 392483.0 |
| share | 750 | 0.0 | 2.2 | 8.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2016 | 여자 | 경영·회계·사무 관련직 | 29.6 |
| 2016 | 전체 | 경영·회계·사무 관련직 | 26.4 |
| 2016 | 남자 | 경영·회계·사무 관련직 | 23.1 |
| 2016 | 여자 | 보건·의료 관련직 | 17.2 |
| 2016 | 여자 | 교육 및 자연과학·사회과학 연구관련직 | 11.8 |

![GOMS_007 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_007_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_007_취업자의_성별_직업_분포2016_freq_share.xls")


### GOMS_008 — 취업자의 성별 직업 분포2017

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_008_취업자의_성별_직업_분포2017_freq_share.xls` |
| 원본 표 크기 | 11행 x 19열 |
| long 변환 행수 | 198행 |
| 연도 범위 | 2017~2019 (3개 연도) |
| 하위그룹 수 | 3 |
| 항목 수 | 11 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 99 | 0.0 | 18933.0 | 42904.7 | 375185.0 |
| share | 99 | 0.0 | 8.6 | 18.1 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 여자 | 경영·사무·금융·보험직 | 31.0 |
| 2019 | 전체 | 경영·사무·금융·보험직 | 28.8 |
| 2019 | 남자 | 경영·사무·금융·보험직 | 26.6 |
| 2019 | 남자 | 연구직 및 공학 기술직 | 26.1 |
| 2019 | 여자 | 보건·의료직 | 19.9 |

![GOMS_008 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_008_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_008_취업자의_성별_직업_분포2017_freq_share.xls")


### GOMS_009 — 취업자의 학교유형별 직업 분포2016

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_009_취업자의_학교유형별_직업_분포2016_freq_share.xls` |
| 원본 표 크기 | 25행 x 81열 |
| long 변환 행수 | 2,000행 |
| 연도 범위 | 2007~2016 (10개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 25 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 1,000 | 0.0 | 2845.5 | 15118.7 | 392483.0 |
| share | 1,000 | 0.0 | 1.1 | 8.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2016 | 교육대 | 교육 및 자연과학·사회과학 연구관련직 | 96.9 |
| 2016 | 4년제 | 경영·회계·사무 관련직 | 30.5 |
| 2016 | 전체 | 경영·회계·사무 관련직 | 26.4 |
| 2016 | 전문대 | 경영·회계·사무 관련직 | 19.5 |
| 2016 | 전문대 | 보건·의료 관련직 | 17.5 |

![GOMS_009 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_009_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_009_취업자의_학교유형별_직업_분포2016_freq_share.xls")


### GOMS_010 — 취업자의 학교유형별 직업 분포2017

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_010_취업자의_학교유형별_직업_분포2017_freq_share.xls` |
| 원본 표 크기 | 11행 x 25열 |
| long 변환 행수 | 264행 |
| 연도 범위 | 2017~2019 (3개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 11 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 132 | 0.0 | 12178.5 | 32178.5 | 375185.0 |
| share | 132 | 0.0 | 7.5 | 18.1 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 교육대 | 교육·법률·사회복지·경찰·소방직 및 군인 | 99.6 |
| 2019 | 4년제 | 경영·사무·금융·보험직 | 34.5 |
| 2019 | 전체 | 경영·사무·금융·보험직 | 28.8 |
| 2019 | 4년제 | 연구직 및 공학 기술직 | 21.3 |
| 2019 | 전문대 | 보건·의료직 | 21.2 |

![GOMS_010 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_010_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_010_취업자의_학교유형별_직업_분포2017_freq_share.xls")


### GOMS_011 — 취업자의 전공별 직업 분포2016

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_011_취업자의_전공별_직업_분포2016_freq_share.xls` |
| 원본 표 크기 | 25행 x 161열 |
| long 변환 행수 | 4,000행 |
| 연도 범위 | 2007~2016 (10개 연도) |
| 하위그룹 수 | 8 |
| 항목 수 | 25 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 2,000 | 0.0 | 709.5 | 7559.1 | 392483.0 |
| share | 2,000 | 0.0 | 1.1 | 8.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2016 | 의약 | 보건·의료 관련직 | 82.1 |
| 2016 | 교육 | 교육 및 자연과학·사회과학 연구관련직 | 55.6 |
| 2016 | 사회 | 경영·회계·사무 관련직 | 46.4 |
| 2016 | 인문 | 경영·회계·사무 관련직 | 45.4 |
| 2016 | 예체능 | 문화·예술·디자인·방송 관련직 | 28.2 |

![GOMS_011 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_011_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_011_취업자의_전공별_직업_분포2016_freq_share.xls")


### GOMS_012 — 취업자의 전공별 직업 분포2017

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_012_취업자의_전공별_직업_분포2017_freq_share.xls` |
| 원본 표 크기 | 11행 x 49열 |
| long 변환 행수 | 528행 |
| 연도 범위 | 2017~2019 (3개 연도) |
| 하위그룹 수 | 8 |
| 항목 수 | 11 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 264 | 0.0 | 2461.5 | 16088.9 | 375185.0 |
| share | 264 | 0.0 | 5.7 | 18.1 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 의약 | 보건·의료직 | 84.2 |
| 2019 | 교육 | 교육·법률·사회복지·경찰·소방직 및 군인 | 79.4 |
| 2019 | 사회 | 경영·사무·금융·보험직 | 55.1 |
| 2019 | 인문 | 경영·사무·금융·보험직 | 51.7 |
| 2019 | 공학 | 연구직 및 공학 기술직 | 49.3 |

![GOMS_012 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_012_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_012_취업자의_전공별_직업_분포2017_freq_share.xls")


### GOMS_013 — 취업자의 성별 사업체 규모

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_013_취업자의_성별_사업체_규모_freq_share.xls` |
| 원본 표 크기 | 10행 x 79열 |
| long 변환 행수 | 780행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 3 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 390 | 6508.0 | 24219.0 | 48305.3 | 391857.0 |
| share | 390 | 3.6 | 11.4 | 20.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 여자 | 3. 10～29명 | 21.7 |
| 2019 | 전체 | 3. 10～29명 | 19.6 |
| 2019 | 남자 | 3. 10～29명 | 17.5 |
| 2019 | 남자 | 9. 1,000명 이상 | 16.0 |
| 2019 | 남자 | 6. 100～299명 | 14.4 |

![GOMS_013 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_013_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_013_취업자의_성별_사업체_규모_freq_share.xls")


### GOMS_014 — 취업자의 학교유형별 사업체 규모

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_014_취업자의_학교유형별_사업체_규모_freq_share.xls` |
| 원본 표 크기 | 10행 x 105열 |
| long 변환 행수 | 1,040행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 520 | 0.0 | 19393.5 | 36228.9 | 391857.0 |
| share | 520 | 0.0 | 11.0 | 20.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 교육대 | 5. 50～99명 | 44.9 |
| 2019 | 교육대 | 4. 30～49명 | 26.2 |
| 2019 | 전문대 | 3. 10～29명 | 24.6 |
| 2019 | 교육대 | 3. 10～29명 | 23.1 |
| 2019 | 전체 | 3. 10～29명 | 19.6 |

![GOMS_014 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_014_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_014_취업자의_학교유형별_사업체_규모_freq_share.xls")


### GOMS_015 — 취업자의 전공별 사업체 규모

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_015_취업자의_전공별_사업체_규모_freq_share.xls` |
| 원본 표 크기 | 10행 x 209열 |
| long 변환 행수 | 2,080행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 8 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 1,040 | 130.0 | 5618.5 | 18114.2 | 391857.0 |
| share | 1,040 | 0.6 | 11.2 | 20.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 교육 | 3. 10～29명 | 35.3 |
| 2019 | 예체능 | 1. 1～4명 | 28.6 |
| 2019 | 의약 | 9. 1,000명 이상 | 22.1 |
| 2019 | 사회 | 3. 10～29명 | 21.7 |
| 2019 | 예체능 | 2. 5～9명 | 21.7 |

![GOMS_015 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_015_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_015_취업자의_전공별_사업체_규모_freq_share.xls")


### GOMS_016 — 취업자의 성별 사업체 유형

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_016_취업자의_성별_사업체_유형_freq_share.xls` |
| 원본 표 크기 | 10행 x 79열 |
| long 변환 행수 | 780행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 3 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 390 | 0.0 | 9654.5 | 49682.5 | 391086.0 |
| share | 390 | 0.0 | 4.2 | 20.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 남자 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 66.0 |
| 2019 | 전체 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 62.8 |
| 2019 | 여자 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 59.7 |
| 2019 | 여자 | 6. 교육기관(대학, 초/중/고 등) | 12.1 |
| 2019 | 여자 | 4. (재단, 사단) 법인단체 | 11.6 |

![GOMS_016 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_016_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_016_취업자의_성별_사업체_유형_freq_share.xls")


### GOMS_017 — 취업자의 학교유형별 사업체 유형

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_017_취업자의_학교유형별_사업체_유형_freq_share.xls` |
| 원본 표 크기 | 10행 x 105열 |
| long 변환 행수 | 1,040행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 520 | 0.0 | 4800.5 | 37261.8 | 391086.0 |
| share | 520 | 0.0 | 3.2 | 20.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 교육대 | 6. 교육기관(대학, 초/중/고 등) | 97.8 |
| 2019 | 전문대 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 70.9 |
| 2019 | 전체 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 62.8 |
| 2019 | 4년제 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 59.5 |
| 2019 | 전문대 | 4. (재단, 사단) 법인단체 | 10.5 |

![GOMS_017 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_017_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_017_취업자의_학교유형별_사업체_유형_freq_share.xls")


### GOMS_018 — 취업자의 전공별 사업체 유형

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_018_취업자의_전공별_사업체_유형_freq_share.xls` |
| 원본 표 크기 | 10행 x 209열 |
| long 변환 행수 | 2,080행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 8 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 1,040 | 0.0 | 2068.5 | 18630.6 | 391086.0 |
| share | 1,040 | 0.0 | 3.9 | 20.0 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 예체능 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 76.2 |
| 2019 | 공학 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 74.0 |
| 2019 | 교육 | 6. 교육기관(대학, 초/중/고 등) | 63.8 |
| 2019 | 자연 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 63.1 |
| 2019 | 전체 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 62.8 |

![GOMS_018 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_018_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_018_취업자의_전공별_사업체_유형_freq_share.xls")


### GOMS_019 — 취업자의 성별 종사상 지위

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_019_취업자의_성별_종사상_지위_freq_share.xls` |
| 원본 표 크기 | 7행 x 79열 |
| long 변환 행수 | 546행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 3 |
| 항목 수 | 7 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 273 | 0.0 | 6677.0 | 70330.2 | 389213.0 |
| share | 273 | 0.0 | 2.6 | 28.5 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 남자 | 1. 상용근로자 | 81.5 |
| 2019 | 전체 | 1. 상용근로자 | 78.9 |
| 2019 | 여자 | 1. 상용근로자 | 76.4 |
| 2019 | 여자 | 2. 임시근로자 | 18.9 |
| 2019 | 전체 | 2. 임시근로자 | 16.0 |

![GOMS_019 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_019_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_019_취업자의_성별_종사상_지위_freq_share.xls")


### GOMS_020 — 취업자의 학교유형별 종사상 지위

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_020_취업자의_학교유형별_종사상_지위_freq_share.xls` |
| 원본 표 크기 | 7행 x 105열 |
| long 변환 행수 | 728행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 4 |
| 항목 수 | 7 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 364 | 0.0 | 4282.5 | 52747.6 | 389213.0 |
| share | 364 | 0.0 | 2.6 | 28.5 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 교육대 | 1. 상용근로자 | 89.9 |
| 2019 | 4년제 | 1. 상용근로자 | 79.4 |
| 2019 | 전체 | 1. 상용근로자 | 78.9 |
| 2019 | 전문대 | 1. 상용근로자 | 77.7 |
| 2019 | 전문대 | 2. 임시근로자 | 16.0 |

![GOMS_020 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_020_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_020_취업자의_학교유형별_종사상_지위_freq_share.xls")


### GOMS_021 — 취업자의 전공별 종사상 지위

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `frequency_share/GOMS_021_취업자의_전공별_종사상_지위_freq_share.xls` |
| 원본 표 크기 | 7행 x 209열 |
| long 변환 행수 | 1,456행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 8 |
| 항목 수 | 7 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| frequency | 728 | 0.0 | 2695.0 | 26373.5 | 389213.0 |
| share | 728 | 0.0 | 2.7 | 28.5 | 100.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 비중(%) |
| --- | --- | --- | ---: |
| 2019 | 의약 | 1. 상용근로자 | 89.7 |
| 2019 | 공학 | 1. 상용근로자 | 84.8 |
| 2019 | 교육 | 1. 상용근로자 | 80.8 |
| 2019 | 전체 | 1. 상용근로자 | 78.9 |
| 2019 | 사회 | 1. 상용근로자 | 78.5 |

![GOMS_021 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_021_frequency_share.png)


In [ ]:
show_goms_raw_file("frequency_share/GOMS_021_취업자의_전공별_종사상_지위_freq_share.xls")


### GOMS_022 — 성별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_022_성별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 3행 x 14열 |
| long 변환 행수 | 39행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 3 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 39 | 161.3 | 207.9 | 207.9 | 259.4 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 남자 | 259.4 |
| 2019 | 평균 | 여자 | 226.4 |

![GOMS_022 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_022_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_022_성별_월평균_근로소득_mean.xls")


### GOMS_023 — 학교유형별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_023_학교유형별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 4행 x 14열 |
| long 변환 행수 | 52행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 4 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 52 | 172.3 | 207.2 | 208.7 | 250.4 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 4년제 | 250.4 |
| 2019 | 평균 | 교육대 | 248.3 |
| 2019 | 평균 | 전문대 | 214.9 |

![GOMS_023 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_023_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_023_학교유형별_월평균_근로소득_mean.xls")


### GOMS_024 — 전공계열별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_024_전공계열별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 8행 x 14열 |
| long 변환 행수 | 104행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 8 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 104 | 147.7 | 204.3 | 205.1 | 271.8 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 의약 | 271.8 |
| 2019 | 평균 | 인문 | 269.8 |
| 2019 | 평균 | 공학 | 260.6 |
| 2019 | 평균 | 사회 | 234.6 |
| 2019 | 평균 | 자연 | 225.2 |

![GOMS_024 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_024_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_024_전공계열별_월평균_근로소득_mean.xls")


### GOMS_025 — 산업별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_025_산업별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 22행 x 14열 |
| long 변환 행수 | 286행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 22 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 286 | 0.0 | 212.5 | 212.9 | 523.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 금융 및 보험업 | 523.0 |
| 2019 | 평균 | 전기, 가스, 증기 및 수도사업 | 326.0 |
| 2019 | 평균 | 광업 | 305.0 |
| 2019 | 평균 | 제조업 | 283.8 |
| 2019 | 평균 | 건설업 | 276.0 |

![GOMS_025 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_025_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_025_산업별_월평균_근로소득_mean.xls")


### GOMS_026 — 직업별 월평균 근로소득2016

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_026_직업별_월평균_근로소득2016_mean.xls` |
| 원본 표 크기 | 25행 x 11열 |
| long 변환 행수 | 250행 |
| 연도 범위 | 2007~2016 (10개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 25 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 250 | 131.1 | 207.2 | 219.7 | 640.2 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2016 | 평균 | 관리직 | 603.2 |
| 2016 | 평균 | 금융·보험 관련직 | 298.5 |
| 2016 | 평균 | 농림어업 관련직 | 289.8 |
| 2016 | 평균 | 법률·경찰·소방·교도 관련직 | 270.2 |
| 2016 | 평균 | 기계 관련직 | 265.9 |

![GOMS_026 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_026_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_026_직업별_월평균_근로소득2016_mean.xls")


### GOMS_027 — 직업별 월평균 근로소득2017

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_027_직업별_월평균_근로소득2017_mean.xls` |
| 원본 표 크기 | 11행 x 4열 |
| long 변환 행수 | 33행 |
| 연도 범위 | 2017~2019 (3개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 11 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 33 | 148.3 | 244.2 | 237.0 | 328.8 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 건설·채굴직 | 302.7 |
| 2019 | 평균 | 보건·의료직 | 274.1 |
| 2019 | 평균 | 설치·정비·생산직 | 273.6 |
| 2019 | 평균 | 연구직 및 공학 기술직 | 259.2 |
| 2019 | 평균 | 농림어업직 | 249.6 |

![GOMS_027 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_027_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_027_직업별_월평균_근로소득2017_mean.xls")


### GOMS_028 — 사업체 규모별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_028_사업체_규모별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 10행 x 14열 |
| long 변환 행수 | 130행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 130 | 160.0 | 208.1 | 213.9 | 360.2 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 9. 1,000명 이상 | 360.2 |
| 2019 | 평균 | 8. 500～999명 | 273.1 |
| 2019 | 평균 | 7. 300～499명 | 270.2 |
| 2019 | 평균 | 6. 100～299명 | 254.0 |
| 2019 | 평균 | 5. 50～99명 | 243.8 |

![GOMS_028 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_028_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_028_사업체_규모별_월평균_근로소득_mean.xls")


### GOMS_029 — 사업체 유형별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_029_사업체_유형별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 10행 x 14열 |
| long 변환 행수 | 130행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 130 | 100.3 | 198.7 | 198.0 | 283.3 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 2. 외국인이 운영하는 국내외에 위치하는 외국인 회사 | 269.5 |
| 2019 | 평균 | 3. 정부투자기관/정부출연기관/공사합동기업 | 267.5 |
| 2019 | 평균 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 251.3 |
| 2019 | 평균 | 4. (재단, 사단) 법인단체 | 247.2 |
| 2019 | 평균 | 8. 특정한 회사나 사업체에 소속되어 있지 않다 | 242.2 |

![GOMS_029 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_029_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_029_사업체_유형별_월평균_근로소득_mean.xls")


### GOMS_030 — 종사상지위별 월평균 근로소득

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_030_종사상지위별_월평균_근로소득_mean.xls` |
| 원본 표 크기 | 7행 x 14열 |
| long 변환 행수 | 91행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 7 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 91 | 0.0 | 193.6 | 182.1 | 461.8 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 4. 고용원이 있는 자영업자 | 461.8 |
| 2019 | 평균 | 1. 상용근로자 | 264.6 |
| 2019 | 평균 | 5. 고용원이 없는 자영업자 | 196.0 |
| 2019 | 평균 | 2. 임시근로자 | 146.3 |
| 2019 | 평균 | 3. 일용근로자 | 117.3 |

![GOMS_030 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_030_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_030_종사상지위별_월평균_근로소득_mean.xls")


### GOMS_031 — 성별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_031_성별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 3행 x 14열 |
| long 변환 행수 | 39행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 3 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 39 | 40.0 | 44.4 | 44.5 | 49.2 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 남자 | 43.8 |
| 2019 | 평균 | 여자 | 40.0 |

![GOMS_031 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_031_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_031_성별_주당_평균_근로시간_mean.xls")


### GOMS_032 — 학교유형별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_032_학교유형별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 4행 x 14열 |
| long 변환 행수 | 52행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 4 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 52 | 40.2 | 44.5 | 44.1 | 48.3 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 전문대 | 42.1 |
| 2019 | 평균 | 4년제 | 41.8 |
| 2019 | 평균 | 교육대 | 40.6 |

![GOMS_032 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_032_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_032_학교유형별_주당_평균_근로시간_mean.xls")


### GOMS_033 — 전공계열별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_033_전공계열별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 8행 x 14열 |
| long 변환 행수 | 104행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 8 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 104 | 37.6 | 44.3 | 44.2 | 50.4 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 의약 | 44.8 |
| 2019 | 평균 | 공학 | 43.7 |
| 2019 | 평균 | 자연 | 42.5 |
| 2019 | 평균 | 사회 | 41.9 |
| 2019 | 평균 | 인문 | 39.9 |

![GOMS_033 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_033_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_033_전공계열별_주당_평균_근로시간_mean.xls")


### GOMS_034 — 산업별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_034_산업별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 22행 x 14열 |
| long 변환 행수 | 286행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 22 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 286 | 0.0 | 45.1 | 43.7 | 72.0 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 농업,임업 및 어업 | 46.9 |
| 2019 | 평균 | 건설업 | 46.9 |
| 2019 | 평균 | 하수·폐기물처리, 원료재생 및 환경복원업 | 45.7 |
| 2019 | 평균 | 제조업 | 44.9 |
| 2019 | 평균 | 공공행정, 국방 및 사회보장 행정 | 44.3 |

![GOMS_034 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_034_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_034_산업별_주당_평균_근로시간_mean.xls")


### GOMS_035 — 직업별 주당 평균 근로시간2016

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_035_직업별_주당_평균_근로시간2016_mean.xls` |
| 원본 표 크기 | 25행 x 11열 |
| long 변환 행수 | 250행 |
| 연도 범위 | 2007~2016 (10개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 25 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 250 | 36.9 | 47.2 | 47.0 | 57.1 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2016 | 평균 | 재료관련직 | 49.9 |
| 2016 | 평균 | 섬유 및 의복 관련직 | 49.3 |
| 2016 | 평균 | 법률·경찰·소방·교도 관련직 | 49.1 |
| 2016 | 평균 | 군인 | 48.6 |
| 2016 | 평균 | 식품가공 관련직 | 48.5 |

![GOMS_035 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_035_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_035_직업별_주당_평균_근로시간2016_mean.xls")


### GOMS_036 — 직업별 주당 평균 근로시간2017

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_036_직업별_주당_평균_근로시간2017_mean.xls` |
| 원본 표 크기 | 11행 x 4열 |
| long 변환 행수 | 33행 |
| 연도 범위 | 2017~2019 (3개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 11 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 33 | 33.6 | 42.9 | 42.7 | 49.6 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 건설·채굴직 | 46.2 |
| 2019 | 평균 | 농림어업직 | 46.1 |
| 2019 | 평균 | 설치·정비·생산직 | 45.8 |
| 2019 | 평균 | 보건·의료직 | 45.3 |
| 2019 | 평균 | 연구직 및 공학 기술직 | 44.7 |

![GOMS_036 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_036_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_036_직업별_주당_평균_근로시간2017_mean.xls")


### GOMS_037 — 사업체 규모별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_037_사업체_규모별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 10행 x 14열 |
| long 변환 행수 | 130행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 130 | 37.2 | 45.0 | 44.8 | 48.6 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 7. 300～499명 | 44.9 |
| 2019 | 평균 | 9. 1,000명 이상 | 44.5 |
| 2019 | 평균 | 8. 500～999명 | 44.2 |
| 2019 | 평균 | 6. 100～299명 | 44.0 |
| 2019 | 평균 | 5. 50～99명 | 42.9 |

![GOMS_037 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_037_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_037_사업체_규모별_주당_평균_근로시간_mean.xls")


### GOMS_038 — 사업체 유형별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_038_사업체_유형별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 10행 x 14열 |
| long 변환 행수 | 130행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 10 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 130 | 27.3 | 43.6 | 42.2 | 48.5 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 5. 정부기관(공무원, 군인 등) | 44.3 |
| 2019 | 평균 | 4. (재단, 사단) 법인단체 | 42.9 |
| 2019 | 평균 | 9. 기타 | 42.7 |
| 2019 | 평균 | 3. 정부투자기관/정부출연기관/공사합동기업 | 42.3 |
| 2019 | 평균 | 1. 내국인이 운영하는 국내외의 민간회사 또는개인사업체 | 41.8 |

![GOMS_038 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_038_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_038_사업체_유형별_주당_평균_근로시간_mean.xls")


### GOMS_039 — 종사상지위별 주당 평균 근로시간

| 항목 | 값 |
| --- | --- |
| 원본 파일 | `mean/GOMS_039_종사상지위별_주당_평균_근로시간_mean.xls` |
| 원본 표 크기 | 7행 x 14열 |
| long 변환 행수 | 91행 |
| 연도 범위 | 2007~2019 (13개 연도) |
| 하위그룹 수 | 1 |
| 항목 수 | 7 |

**측정값 기술통계**

| measure_type | count | min | median | mean | max |
| --- | ---: | ---: | ---: | ---: | ---: |
| mean | 91 | 25.0 | 43.1 | 42.2 | 57.1 |

**최근 연도 상위값 미리보기**

| 연도 | 하위그룹 | 항목 | 평균값 |
| --- | --- | --- | ---: |
| 2019 | 평균 | 4. 고용원이 있는 자영업자 | 48.4 |
| 2019 | 평균 | 1. 상용근로자 | 44.1 |
| 2019 | 평균 | 6. 무급가족종사자 | 39.0 |
| 2019 | 평균 | 5. 고용원이 없는 자영업자 | 38.8 |
| 2019 | 평균 | 2. 임시근로자 | 33.0 |

![GOMS_039 visualization](P2_G1_kedi_raw_download_visuals/figures/GOMS_039_mean.png)


In [ ]:
show_goms_raw_file("mean/GOMS_039_종사상지위별_주당_평균_근로시간_mean.xls")
